In [1]:
%load_ext dotenv
%dotenv

In [ ]:
from graphrag.graph_utils import get_neo4j_driver, init_neo4j_constraints, save_chapter, insert_chunks 
from graphrag.gen_utils import initialize_nltk, create_chunks
from graphrag.war_and_peace_schemas import get_schema_entities, entity_types_map, relationship_types_map

import json
import requests

from tqdm import tqdm
from typing import List, Dict, Optional, Literal, Annotated, Union
from pydantic import BaseModel, Field, ConfigDict

initialize_nltk()
neo4j_driver = get_neo4j_driver()
init_neo4j_constraints(neo4j_driver, get_schema_entities())

#### The following parsing logic applies specifically to Leo Tolstoy organization of War and Peace into books and chapters and to this specific version from Guttenberg project.

In [2]:
import re

BOOK_START = "BOOK ONE: 1805"
BOOK_END = "*** END OF THE PROJECT GUTENBERG EBOOK WAR AND PEACE ***"

try:
  url = "https://www.gutenberg.org/files/2600/2600-0.txt"
  response = requests.get(url, timeout=15)
  response.raise_for_status()
  text = response.text
  print("Successfully loaded text from Project Gutenberg.")
except requests.RequestException as e:
  print(f"URL access failed: {e}")
  print("Loading text from local file instead.")
  with open("../data/war_and_peace.txt", "r", encoding="utf-8") as file:
    text = file.read()

def get_chapters(text: str) -> List[Dict[str, str]]:
  pattern = r"(\s*CHAPTER\s+[IVXLCDM]+)\b\s+"
  parts = re.split(pattern, text, flags=re.IGNORECASE)
  return [{"title": parts[i].strip(), "content": parts[i + 1].strip()} for i in range(1, len(parts) - 1, 2)]

def chunk_into_books(text: str, start: str = None, end: str = None) -> List[Dict[str, any]]:
  if start is not None:
    text = f"{start}\n{text.split(start)[-1]}"
  if end is not None:
    text = text.split(end)[0]
  pattern = r"(Book [A-Z]+:\s*[^\r\n]*)(?=\s+Chapter\s+I\b)"
  parts = re.split(pattern, text, flags=re.IGNORECASE)
  return [{"title": parts[i].strip(), "chapters": get_chapters(parts[i + 1].strip())} 
           for i in range(1, len(parts) - 1, 2)]

books = chunk_into_books(text, start=BOOK_START, end=BOOK_END)

URL access failed: 503 Server Error: Service Unavailable for url: https://www.gutenberg.org/files/2600/2600-0.txt
Loading text from local file instead.


In [ ]:
from graphrag.war_and_peace_schemas import (Person, Family, Organization, Location, MilitaryUnit, Event, 
                                            TitleOrRank, Concept, EntityResolutionResult)
from graphrag.llm import get_openai_client, SYSTEM_CONSOLIDATION_PROMPT

client = get_openai_client()
   
def entity_consolidator (raw_entities):
  entities = []
  for item in raw_entities.values():
    entity = item["entity"]
    entities.append({"id": item["_id_"],
                     "entity_type": entity.entity_type,
                     "name": entity.name,
                     "source_quote": entity.source_quote,
                     "attributes": entity.model_dump(mode="json", exclude={"name", "entity_type", "source_quote"}, exclude_none=True)})

  response = client.chat.completions.parse(model="gpt-oss:20b",
                                           temperature=0,
                                           reasoning_effort="low",
                                           max_completion_tokens=4096,
                                           messages=[{"role": "system", "content": SYSTEM_CONSOLIDATION_PROMPT},
                                                     {"role": "user", "content": json.dumps(entities)}],
                                           response_format=EntityResolutionResult)                                        
  return response.choices[0].message.parsed.model_dump(mode="json")["merge_groups"]

In [ ]:
from graphrag.llm import create_extraction_prompt, GRAPH_EXTRACTION_SYSTEM_PROMPT
from graphrag.war_and_peace_schemas import ExtractionResult, Person, EntityRecord, Family, Location
import time

chunk_id = 34 # 0
for book_ndx, book in enumerate(tqdm(books, desc="Reading books")):
  # Only loading first book
  if book_ndx > 0:
    break
  print(f"Reading book # {book_ndx+1}, {book["title"]}...")
  for chapter_ndx, chapter in enumerate(tqdm(book["chapters"][4:], desc="Reading book chapters"), start=4):
    # Only loading 5 chapters
    if chapter_ndx > 4:
      break
    chunks = create_chunks(chapter["content"])
    insert_chunks (neo4j_driver, book["title"], chapter["title"], chunks, chunk_id+1)
    chunks_result : dict[int, ExtractionResult] = {}
    for chunk_ndx, chunk in enumerate(tqdm(chunks, desc="Processing chunks")):
      chunk_id += 1
      print(f"Processing chunk # {chunk_ndx+1}/{len(chunks)} from book # {book_ndx+1}/{len(books)} {book["title"]}, "
            f"chapter # {chapter_ndx+1}/{len(book["chapters"])} {chapter["title"]}")
      user_prompt = create_extraction_prompt(chunk)
      response = client.chat.completions.parse(model="gpt-oss:20b",
                                               temperature=0,
                                               reasoning_effort="low",
                                               max_completion_tokens=4096,
                                               messages=[{"role": "system", "content": GRAPH_EXTRACTION_SYSTEM_PROMPT},
                                                         {"role": "user", "content": user_prompt}],
                                               response_format=ExtractionResult)
      print(f"Chunk # {chunk_ndx+1}, with {response.choices[0].finish_reason} finish reason::\n"
            f"  - {response.usage.prompt_tokens} tokens App--> LLM; "
            f"{response.usage.completion_tokens} tokens LLM--> App in {response.usage.total_tokens} total tokens\n"
            f"  - Results: {response.choices[0].message.parsed}")
      chunks_result[chunk_id] = response.choices[0].message.parsed
      time.sleep(30)
    entities, relationships = save_chapter(neo4j_driver, entity_consolidator, chunks_result, entity_types_map, relationship_types_map)
    print(f"Chapter entities summary: {entities}")
    print(f"Chapter relationship summary: {relationships}")
    time.sleep(60)

Reading books:   0%|          | 0/15 [00:00<?, ?it/s]

Reading book # 1, BOOK ONE: 1805...


Processing chunk # 1/10 from book # 1/15 BOOK ONE: 1805, chapter # 5/28 CHAPTER V
Chunk # 1, with stop finish reason::
  - 9416 tokens App--> LLM; 1793 tokens LLM--> App in 11209 total tokens
  - Results: entities=[Person(name='Anna Pávlovna', description='A woman who is questioned about a comedy and speaks about petitions and sovereigns.', source_quote='asked Anna Pávlovna', entity_type='PERSON', aliases=None, gender='female', occupation=None, nationality=None, evidence_sentences=['And what do you think of this latest comedy, the coronation at Milan?” asked Anna Pávlovna', 'Prince Andrew looked Anna Pávlovna straight in the face with a sarcastic smile.']), Person(name='Prince Andrew', description='A man who looks at Anna Pávlovna with a sarcastic smile.', source_quote='Prince Andrew', entity_type='PERSON', aliases=None, gender='male', occupation=None, nationality=None, evidence_sentences=['Prince Andrew looked Anna Pávlovna straight in the face with a sarcastic smile.']), Person(name=

Processing chunk # 2/10 from book # 1/15 BOOK ONE: 1805, chapter # 5/28 CHAPTER V
Chunk # 2, with stop finish reason::
  - 9360 tokens App--> LLM; 1067 tokens LLM--> App in 10427 total tokens
  - Results: entities=[Person(name='Hippolyte', description=None, source_quote='Prince Hippolyte', entity_type='PERSON', aliases=None, gender='male', occupation=None, nationality=None, evidence_sentences=['Prince Hippolyte, who had been gazing at the vicomte for some time through his lorgnette, suddenly turned completely round toward the little princess, and having asked for a needle began tracing the Condé coat of arms on the table.']), Person(name='Vicomte', description=None, source_quote='vicomte', entity_type='PERSON', aliases=None, gender=None, occupation=None, nationality=None, evidence_sentences=['Prince Hippolyte, who had been gazing at the vicomte for some time through his lorgnette, suddenly turned completely round toward the little princess, and having asked for a needle began tracing t

Processing chunk # 3/10 from book # 1/15 BOOK ONE: 1805, chapter # 5/28 CHAPTER V
Chunk # 3, with stop finish reason::
  - 9392 tokens App--> LLM; 965 tokens LLM--> App in 10357 total tokens
  - Results: entities=[Person(name='Pierre', description=None, source_quote='Pierre wished to make a remark', entity_type='PERSON', aliases=None, gender=None, occupation=None, nationality=None, evidence_sentences=['Pierre wished to make a remark, for the conversation interested him']), Person(name='Anna Pávlovna', description=None, source_quote='Anna Pávlovna, who had him under observation, interrupted', entity_type='PERSON', aliases=None, gender=None, occupation=None, nationality=None, evidence_sentences=['Anna Pávlovna, who had him under observation, interrupted']), Person(name='Emperor Alexander', description=None, source_quote='The Emperor Alexander', entity_type='PERSON', aliases=None, gender='male', occupation=None, nationality=None, evidence_sentences=['The Emperor Alexander']), Person(name=

Processing chunk # 4/10 from book # 1/15 BOOK ONE: 1805, chapter # 5/28 CHAPTER V
Chunk # 4, with stop finish reason::
  - 9413 tokens App--> LLM; 1738 tokens LLM--> App in 11151 total tokens
  - Results: entities=[Person(name='Andrew', description='A prince who remarks sarcastically about the vicomte.', source_quote='Prince Andrew', entity_type='PERSON', aliases=[], gender='male', occupation=None, nationality=None, evidence_sentences=['Bonaparte has said so,” remarked Prince Andrew with a sarcastic smile.', 'Prince Andrew continued after a short silence, again quoting Napoleon’s words.']), Person(name='Anna Pávlovna', description='A woman who is addressed by Pierre and feels uneasy about his remarks.', source_quote='Anna Pávlovna', entity_type='PERSON', aliases=[], gender='female', occupation=None, nationality=None, evidence_sentences=['turning to Anna Pávlovna, “he ever was a hero, after the murder of the duc there was one martyr more in heaven and one hero less on earth.”', 'though 

... (7 outputs left out)

entities_by_key/1: {('PERSON', 'anna pávlovna'): {'_id_': 1, 'entity': Person(name='Anna Pávlovna', description='A woman who is questioned about a comedy and speaks about petitions and sovereigns.', source_quote='asked Anna Pávlovna', entity_type='PERSON', aliases=[], gender='female', occupation=None, nationality=None, evidence_sentences=['And what do you think of this latest comedy, the coronation at Milan?” asked Anna Pávlovna', 'Prince Andrew looked Anna Pávlovna straight in the face with a sarcastic smile.', 'Anna Pávlovna, who had him under observation, interrupted', 'turning to Anna Pávlovna, “he ever was a hero, after the murder of the duc there was one martyr more in heaven and one hero less on earth.”', 'though Anna Pávlovna felt sure he would say something inappropriate, she was unable to stop him.', '“Dieu! Mon Dieu!” muttered Anna Pávlovna in a terrified whisper.', 'But Pierre continued his speech without heeding her.', '"But won’t you come to this other table?" repeated An

/mnt/c/Apps/Projects/GraphRAG/.venv/lib/python3.13/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='evidence_sentences', input_value=['“Capital!” said Pri...rked Prince Hippolyte.'], input_type=list])
  return self.__pydantic_serializer__.to_python(
/mnt/c/Apps/Projects/GraphRAG/.venv/lib/python3.13/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='evidence_sentences', input_value=['and Monsieur Buonaparte...itions of the nations?'], input_type=list])
  return self.__pydantic_serializer__.to_python(
/mnt/c/Apps/Projects/GraphRAG/.venv/lib/python3.13/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as e

Chapter entities summary: InsertSummary(nodes_created=18, nodes_deleted=0, relationships_created=82, relationships_deleted=0, properties_set=109, labels_added=18)
Chapter relationship summary: InsertSummary(nodes_created=0, nodes_deleted=0, relationships_created=4, relationships_deleted=0, properties_set=7, labels_added=0)


Reading books:   7%|▋         | 1/15 [3:33:06<49:43:24, 12786.04s/it]
